In [7]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Import CRUD module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# MongoDB connection credentials
username = "aacuser"
password = "CS340"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load the Grazioso Salvare logo image file and encode it in base64 format
# so it can be embedded directly into the Dash application layout
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Div([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'width': '200px'}
        ),
        html.H1('SNHU CS-340 Dashboard - Christopher Mooers')
    ], style={'textAlign': 'center'}),
    
    html.Hr(),
    html.Div([
        # Radio buttons allow the user to select which rescue type filter to apply
        dcc.RadioItems(
        id='filter-type',
        # Each option represents a filtering category for the dataset
        options=[
            {'label': 'All', 'value': 'All'},
            {'label': 'Water Rescue', 'value': 'Water'},
            {'label': 'Mountain/Wilderness Rescue', 'value': 'Mountain'},
            {'label': 'Disaster/Individual Tracking', 'value': 'Disaster'},
            {'label': 'Reset', 'value': 'Reset'}
        ],
        # Default value when the dashboard loads
        value='All',
        labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ]),
    
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        # Enable interactive table features for sorting, filtering, pagination,
        # and single-row selection so the selected row can update the map
        editable = False,
        filter_action = "native",
        sort_action = "native",
        sort_mode = "multi",
        column_selectable = False,
        row_selectable = "single",
        row_deletable = False,
        selected_rows = [0],
        page_action = "native",
        page_current = 0,
        page_size = 10
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################
    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    # This callback updates the data table based on the selected filter option.
    # It queries MongoDB using the CRUD module and returns filtered results.

    # Default query returns all records
    query = {}

    if filter_type == "Water":
        # Filter for water rescue dogs
        query = {
            "breed": {"$regex": "Labrador Retriever|Chesapeake Bay Retriever|Newfoundland"}
        }

    elif filter_type == "Mountain":
        # Filter for mountain or wilderness rescue dogs
        query = {
            "breed": {"$regex": "German Shepherd|Alaskan Malamute|Siberian Husky"}
        }

    elif filter_type == "Disaster":
        # Filter for disaster/individual tracking dogs
        query = {
            "breed": {"$regex": "Doberman|Rottweiler|Belgian Malinois"}
        }

    elif filter_type == "Reset" or filter_type == "All":
        # Reset returns all records
        query = {}

    # Retrieve filtered data from MongoDB
    data = pd.DataFrame.from_records(db.read(query))

    # Drop MongoDB ObjectID column if it exists (prevents Dash errors)
    if '_id' in data.columns:
        data.drop(columns=['_id'], inplace=True)

    # Convert DataFrame to dictionary format required by Dash DataTable
    return data.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # Return no graph if there is no table data available yet
    if viewData is None:
        return []

    # Convert the currently displayed table data into a DataFrame
    dff = pd.DataFrame.from_dict(viewData)

    # Return no graph if the filtered DataFrame is empty
    if dff.empty:
        return []

    # Count the top 10 most common breeds in the currently displayed data
    breed_counts = dff['breed'].value_counts().nlargest(10).reset_index()
    breed_counts.columns = ['breed', 'count']

    # Create a bar chart showing the 10 most common breeds
    return [
        dcc.Graph(
            figure=px.bar(
                breed_counts,
                x='breed',
                y='count',
                title='Top 10 Breeds in Current View'
            )
        )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # On initial load, no columns are selected yet,
    # so Dash passes None. We must handle this to avoid errors.
    if selected_columns is None:
        return []
    # Loop through each selected column and apply a highlight color
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://fishmanila-trustphrase-3000.codio.io/proxy/8050/
